# Phase 1 — Retrieval Baseline for LLM-Based API Service Discovery

**Goal**: Beat paper's Sentence-BERT baseline (Hit@20=0.64, MRR@20=0.37) and approach GPT-4 hierarchical (Hit@20=0.74, MRR@20=0.68) **with pure retrieval** — no LLM in the loop yet.

**Stack**: BM25 + BGE-large-en-v1.5 + RRF fusion, evaluated on 50 test queries (paper-style setup).

**Dataset**: `stabletoolbench/ToolEnv2404` (RapidAPI dump — same domain as paper).

**Compute**: Kaggle T4 x2, ~25–30 min end-to-end.

**Pre-reqs**:
- Notebook settings: GPU T4 x2, Internet ON
- Kaggle Secret: `GROQ_API_KEY` (get from https://console.groq.com/keys)

## Cell 1 — Installs

Upgrade `httpx` first to avoid groq SDK `proxies` kwarg incompatibility. ~3 min.

In [ ]:
!pip install -q --upgrade "httpx==0.27.2"
!pip install -q sentence-transformers==3.0.1 faiss-cpu==1.8.0 rank_bm25==0.2.2 \
                "groq==0.11.0" huggingface_hub==0.24.0 datasets==2.20.0 \
                FlagEmbedding==1.2.11
!pip install -q --upgrade transformers==4.44.2
print("installs complete")

## Cell 2 — Imports, seeds, paths

In [ ]:
import os, json, random, pickle, time, tarfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict, Counter

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

WORK_DIR = Path("/kaggle/working")
CACHE_DIR = WORK_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True, parents=True)
print(f"Cache: {CACHE_DIR}")

## Cell 3 — Groq client (with httpx workaround)

Bypasses the `proxies` kwarg bug by passing a pre-built `httpx.Client`.

In [ ]:
from kaggle_secrets import UserSecretsClient
GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")

import httpx
from groq import Groq

http_client = httpx.Client(timeout=60.0, follow_redirects=True)
client = Groq(api_key=GROQ_API_KEY, http_client=http_client)

# Smoke test
resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "say hi in 3 words"}],
    max_tokens=10,
)
print("Groq works:", resp.choices[0].message.content)

## Cell 4 — Download + extract ToolEnv2404

Dataset ships as `toolenv2404_filtered.tar.gz` — we download and extract. ~5–8 min first time, instant on re-run (cached).

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading ToolEnv2404 (~300MB)...")
DATA_DIR = snapshot_download(
    repo_id="stabletoolbench/ToolEnv2404",
    repo_type="dataset",
    local_dir=str(CACHE_DIR / "toolenv"),
    local_dir_use_symlinks=False,
)
print(f"Downloaded to: {DATA_DIR}")

data_root = Path(DATA_DIR)
tar_path = data_root / "toolenv2404_filtered.tar.gz"
extract_dir = data_root / "extracted"
extract_dir.mkdir(exist_ok=True)
marker = extract_dir / ".done"

if not marker.exists():
    print("Extracting tar.gz (~2–3 min)...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=extract_dir)
    marker.touch()
    print("Extraction done.")
else:
    print("Already extracted.")

# Locate actual root (categories live directly under toolenv2404_filtered/)
tools_root = extract_dir / "toolenv2404_filtered"
assert tools_root.exists(), f"Expected {tools_root} to exist"
cats = [p for p in tools_root.iterdir() if p.is_dir()]
print(f"Categories: {len(cats)}")
print("Sample:", [c.name for c in cats[:5]])

## Cell 5 — Parse corpus: `<category>/<tool>.json` → endpoints

Each tool JSON contains an `api_list` of endpoints. We flatten into one `Endpoint` per endpoint for retrieval. ~1–2 min.

In [ ]:
@dataclass
class Endpoint:
    endpoint_id: str
    category: str
    tool_name: str
    tool_description: str
    endpoint_name: str
    endpoint_description: str
    method: str = ""
    parameters: List[Dict] = field(default_factory=list)

    def text_for_embedding(self) -> str:
        parts = [
            f"Category: {self.category}",
            f"API: {self.tool_name}",
            f"API description: {self.tool_description}",
            f"Endpoint: {self.endpoint_name}",
            f"Endpoint description: {self.endpoint_description}",
        ]
        if self.method:
            parts.append(f"Method: {self.method}")
        if self.parameters:
            param_str = ", ".join(p.get("name", "") for p in self.parameters[:10])
            parts.append(f"Parameters: {param_str}")
        return " | ".join(parts)


def parse_toolenv(tools_root: Path) -> List[Endpoint]:
    endpoints: List[Endpoint] = []
    categories = [p for p in tools_root.iterdir() if p.is_dir()]
    bad_json = skipped_no_api = 0

    for cat_dir in tqdm(categories, desc="Parsing"):
        category = cat_dir.name
        for json_file in cat_dir.glob("*.json"):
            try:
                with open(json_file, "r", encoding="utf-8") as f:
                    meta = json.load(f)
            except Exception:
                bad_json += 1
                continue

            tool_name = json_file.stem
            tool_desc = (meta.get("tool_description") or meta.get("description")
                         or meta.get("tool_name", ""))
            api_list = meta.get("api_list") or meta.get("apis") or []
            if not api_list:
                skipped_no_api += 1
                continue

            for api in api_list:
                ep_name = api.get("name") or api.get("api_name") or ""
                ep_desc = (api.get("description")
                           or api.get("api_description") or "")
                if not ep_name:
                    continue
                method = api.get("method", "")
                params = ((api.get("required_parameters", []) or []) +
                          (api.get("optional_parameters", []) or []))
                eid = f"{category}||{tool_name}||{ep_name}"
                endpoints.append(Endpoint(
                    endpoint_id=eid,
                    category=category,
                    tool_name=tool_name,
                    tool_description=(tool_desc or "")[:500],
                    endpoint_name=ep_name,
                    endpoint_description=(ep_desc or "")[:500],
                    method=method,
                    parameters=params,
                ))

    print(f"Parsed {len(endpoints)} endpoints (bad_json={bad_json}, no_api_list={skipped_no_api})")
    return endpoints


# Load from cache if exists, else parse
cache_pkl = CACHE_DIR / "endpoints.pkl"
if cache_pkl.exists():
    with open(cache_pkl, "rb") as f:
        endpoints = pickle.load(f)
    print(f"Loaded {len(endpoints)} endpoints from cache")
else:
    endpoints = parse_toolenv(tools_root)
    with open(cache_pkl, "wb") as f:
        pickle.dump(endpoints, f)
    print(f"Saved {len(endpoints)} endpoints to cache")

print(f"Unique categories: {len(set(e.category for e in endpoints))}")
print(f"Unique tools (APIs): {len(set((e.category, e.tool_name) for e in endpoints))}")
for e in endpoints[:3]:
    print("---"); print(e.endpoint_id); print(e.endpoint_description[:120])

## Cell 6 — Sample 50 test endpoints (10 categories × 5 per category)

Paper's protocol: pick top categories by endpoint count, stratify sampling across diverse tools.

In [ ]:
cat_counts = Counter(e.category for e in endpoints)
top_cats = [c for c, _ in cat_counts.most_common(15)][:10]
print("Selected categories:")
for c in top_cats:
    print(f"  {c}: {cat_counts[c]} endpoints")


def sample_test_endpoints(endpoints, categories, per_cat=5):
    by_cat = defaultdict(list)
    for e in endpoints:
        if e.category in categories and e.endpoint_description.strip():
            by_cat[e.category].append(e)
    test = []
    rng = random.Random(SEED)
    for cat in categories:
        pool = by_cat[cat]
        rng.shuffle(pool)
        seen_tools = set(); picked = []
        for e in pool:
            if e.tool_name not in seen_tools:
                picked.append(e); seen_tools.add(e.tool_name)
            if len(picked) >= per_cat:
                break
        while len(picked) < per_cat and len(pool) > len(picked):
            for e in pool:
                if e not in picked:
                    picked.append(e); break
        test.extend(picked[:per_cat])
    return test


test_endpoints = sample_test_endpoints(endpoints, top_cats, per_cat=5)
print(f"\nTest endpoints: {len(test_endpoints)}")

## Cell 7 — Synthesize user queries via Groq (Llama-3.3-70B)

Paper's exact method: each endpoint description → concise query phrase. ~1 min for 50 queries.

In [ ]:
SYNTH_PROMPT = (
    "Rewrite the following sentence into a concise phrase "
    "(5-10 words) that summarizes its core meaning as a user query. "
    "Return only the phrase, no quotes.\n\nSentence: {desc}"
)


def synthesize_query(desc: str, model="llama-3.3-70b-versatile", retries=3) -> str:
    for i in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": SYNTH_PROMPT.format(desc=desc)}],
                temperature=0.2, max_tokens=40,
            )
            return resp.choices[0].message.content.strip().strip('"').strip("'")
        except Exception as e:
            if i == retries - 1:
                return desc[:80]
            time.sleep(2 ** i)
    return desc[:80]


test_queries_path = CACHE_DIR / "test_queries.json"
if test_queries_path.exists():
    with open(test_queries_path) as f:
        test_queries = json.load(f)
    print(f"Loaded {len(test_queries)} cached test queries")
else:
    test_queries = []
    for e in tqdm(test_endpoints, desc="Synthesizing queries"):
        q = synthesize_query(e.endpoint_description)
        test_queries.append({
            "query": q,
            "gold_endpoint_id": e.endpoint_id,
            "gold_category": e.category,
            "gold_tool": e.tool_name,
            "gold_endpoint": e.endpoint_name,
            "gold_description": e.endpoint_description,
        })
    with open(test_queries_path, "w") as f:
        json.dump(test_queries, f, indent=2)

print("\nSample synthesized queries:")
for t in test_queries[:5]:
    print(f"  Q: {t['query']}")
    print(f"  Gold: {t['gold_description'][:80]}\n")

## Cell 8 — BM25 retriever (lexical baseline)

Tokenize corpus + build BM25 index. ~2 min.

In [ ]:
import re
from rank_bm25 import BM25Okapi


def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


corpus_texts = [e.text_for_embedding() for e in endpoints]
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts, desc="Tokenizing")]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index built over {len(corpus_texts)} docs")


def bm25_search(query: str, top_k: int = 50) -> List[Tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]

## Cell 9 — BGE-large-en-v1.5 dense retriever + FAISS

Biggest single win vs paper's Sentence-BERT baseline. ~10–15 min.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

print("Loading BGE-large-en-v1.5...")
embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device=DEVICE)

embs_path = CACHE_DIR / "corpus_embs.npy"
if embs_path.exists():
    corpus_embs = np.load(embs_path)
    print(f"Loaded cached embeddings: {corpus_embs.shape}")
else:
    print(f"Embedding {len(corpus_texts)} docs (batch=64, fp16)...")
    corpus_embs = embedder.encode(
        corpus_texts,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(embs_path, corpus_embs)

print(f"Shape: {corpus_embs.shape}, dtype: {corpus_embs.dtype}")

index = faiss.IndexFlatIP(corpus_embs.shape[1])
index.add(corpus_embs.astype(np.float32))
print(f"FAISS index size: {index.ntotal}")


def bge_search(query: str, top_k: int = 50) -> List[Tuple[int, float]]:
    q_emb = embedder.encode(
        [f"Represent this sentence for searching relevant passages: {query}"],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    scores, idxs = index.search(q_emb, top_k)
    return list(zip(idxs[0].tolist(), scores[0].tolist()))

## Cell 10 — RRF fusion (BM25 + BGE)

In [ ]:
def rrf_fuse(results_list: List[List[Tuple[int, float]]], k: int = 60, top_k: int = 50) -> List[Tuple[int, float]]:
    scores = defaultdict(float)
    for results in results_list:
        for rank, (doc_id, _score) in enumerate(results):
            scores[doc_id] += 1.0 / (k + rank + 1)
    fused = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
    return fused


def hybrid_search(query: str, top_k: int = 50) -> List[Tuple[int, float]]:
    bm25_res = bm25_search(query, top_k=top_k)
    bge_res = bge_search(query, top_k=top_k)
    return rrf_fuse([bm25_res, bge_res], top_k=top_k)

## Cell 11 — Evaluate: Hit@K, MRR@K + comparison table

Runs BM25, BGE, hybrid on 50 queries; prints side-by-side with paper's numbers.

In [ ]:
def evaluate(retriever_fn, test_queries, endpoints, ks=(5, 10, 20)) -> Dict[str, float]:
    id_to_idx = {e.endpoint_id: i for i, e in enumerate(endpoints)}
    max_k = max(ks)
    hits = {k: 0 for k in ks}
    mrrs = {k: 0.0 for k in ks}
    n = len(test_queries)

    for tq in tqdm(test_queries, desc="Evaluating"):
        gold_idx = id_to_idx.get(tq["gold_endpoint_id"])
        if gold_idx is None:
            continue
        results = retriever_fn(tq["query"], top_k=max_k)
        ranked_ids = [r[0] for r in results]
        rank = None
        for r, doc_id in enumerate(ranked_ids, start=1):
            if doc_id == gold_idx:
                rank = r
                break
        for k in ks:
            if rank is not None and rank <= k:
                hits[k] += 1
                mrrs[k] += 1.0 / rank

    return {f"Hit@{k}": hits[k] / n for k in ks} | {f"MRR@{k}": mrrs[k] / n for k in ks}


print("===== BASELINE: BM25 =====")
res_bm25 = evaluate(bm25_search, test_queries, endpoints)
print(res_bm25)

print("\n===== OURS: BGE-large-en-v1.5 =====")
res_bge = evaluate(bge_search, test_queries, endpoints)
print(res_bge)

print("\n===== OURS: BM25 + BGE RRF =====")
res_hybrid = evaluate(hybrid_search, test_queries, endpoints)
print(res_hybrid)

# Paper's numbers (from Table 3 of Li et al. 2025)
paper_sbert = {"Hit@5": 0.48, "MRR@5": 0.3503, "Hit@10": 0.56, "MRR@10": 0.3609,
               "Hit@20": 0.64, "MRR@20": 0.3668}
paper_gpt4  = {"Hit@5": 0.60, "MRR@5": 0.5073, "Hit@10": 0.66, "MRR@10": 0.5673,
               "Hit@20": 0.74, "MRR@20": 0.6773}

print("\n===== COMPARISON TABLE =====")
print(f"{'Method':<25} {'Hit@5':<8} {'MRR@5':<8} {'Hit@10':<8} {'MRR@10':<8} {'Hit@20':<8} {'MRR@20':<8}")
print("-" * 80)
for name, r in [("Paper SBERT", paper_sbert), ("Paper GPT-4 (target)", paper_gpt4),
                ("Ours BM25", res_bm25), ("Ours BGE-large", res_bge),
                ("Ours BM25+BGE RRF", res_hybrid)]:
    print(f"{name:<25} "
          f"{r['Hit@5']:<8.4f} {r['MRR@5']:<8.4f} "
          f"{r['Hit@10']:<8.4f} {r['MRR@10']:<8.4f} "
          f"{r['Hit@20']:<8.4f} {r['MRR@20']:<8.4f}")

# Save for Phase 2
with open(CACHE_DIR / "phase1_results.json", "w") as f:
    json.dump({"bm25": res_bm25, "bge": res_bge, "hybrid": res_hybrid}, f, indent=2)

print("\nPhase 1 complete. Save notebook + paste the comparison table back.")